# Домашнее задание 7. Сборка конвейера CI/CD
Если у вас еще нет аккаунта в GitLab, вам нужно будет его создать:
1. Перейдите на [GitLab](https://gitlab.com/) и войдите в свой аккаунт.
2. Нажмите на кнопку New Project (Новый проект).
3. Выберите Create blank project (Создать пустой проект).
4. Укажите имя проекта и описание (по желанию).
5. Выберите уровень видимости проекта (Public).
6. Нажмите Create project (Создать проект).
7. Дополните файл .gitlab-ci.yml необходимыми джобами и отправьте в репозиторий.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab




Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Вам дан рабочий код пайплайна и черновик файла .gitlab-ci.yml. Перепишите yaml в [ячейке](#scrollTo=s55MrS66JXWs)


*Ожидаемый артефакт: список коммитов в [ячейке](#scrollTo=gErasBmRSHjb) и ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=F0uQqbe3iHqE)*    

In [ ]:
%%sh
git config --global user.email "you@example.com"
git config --global user.name "Your Name"
git init
pip install scikit-learn numpy pandas -qqq
pip freeze > requirements.txt

Initialized empty Git repository in /content/.git/


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>


In [ ]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Overwriting ml_pipeline.py


### Проверяем работоспособность пайплайна

In [ ]:
!python ml_pipeline.py

Точность аccuracy: 1.00


In [ ]:
%%writefile .gitlab-ci.yml
stages:
  - build
  -

build_model:
  stage: build
  script:
    - echo "запускаем пайплайн..."
    - python ml_pipeline.py


Writing .gitlab-ci.yml


In [ ]:
!git add .gitlab-ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitLab"
!git log

[master cf3cb3c] build(ml_pipeline.py) добавлен пайплайн
 1 file changed, 9 insertions(+)
 create mode 100644 .gitlab-ci.yml
commit cf3cb3c56f0d5674582bdd757066897d653cff31 (HEAD -> master)
Author: Your Name <you@example.com>
Date:   Wed Oct 8 19:26:44 2025 +0000

    build(ml_pipeline.py) добавлен пайплайн

commit c29e7659975b52391b4a07c33a329893f1f8426b
Author: Your Name <you@example.com>
Date:   Wed Oct 8 19:13:08 2025 +0000

    build(ml_pipeline.py) добавлен пайплайн


### Проверка статуса пайплайна

После настройки файла `.gitlab-ci.yml`, вы можете закоммитить изменения и запушить их в репозиторий.

GitLab автоматически запустит пайплайн, и вы сможете наблюдать за его выполнением в разделе CI/CD своего проекта.

Что нужно сделать:

1. Перейдите в свой проект на GitLab.
2. Нажмите на вкладку CI/CD и выберите Pipelines.
3. Вы увидите список запущенных пайплайнов. Нажмите на последний, чтобы увидеть выполнение.
4. Убедитесь, что все джобы выполнены успешно (отмечены зеленым цветом).
5. Приложите ссылку на статус выполнения в разделе Pipelines **своего** репозитория на GitLab.

https://gitverse.ru/Olgas/ml-pipeline/cicd/5

## 2. Обосновать стратегию деплоя (развертывания, Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски




Изучите [инструмент](https://github.com/npryce/adr-tools) для учета архитектурных решений и запишите **причины**, по которым мы начали использовать стратегию деплоя и **риски**, к которым нас привело такое решение.



*Ожидаемый артефакт: архитектурное решение в формате ADR в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

```
Архитектурное решение (ADR): выбор стратегии деплоя для ML-сервиса

Разрабатывается ML-сервис с CI/CD-пайплайном на GitHub Actions/GitVerse. Пайплайн выполняет:
* обучение модели;
* расчёт метрик качества;
* сборку артефактов (модель, метрики).

**Особенности проекта:**
* отсутствие обработки ошибок в коде;
* высокая чувствительность к простоям;
* необходимость минимизации рисков для конечных пользователей при деплое новой версии.

Требуется выбрать стратегию деплоя, которая:
 минимизирует риски сбоев при развёртывании новой версии ML-модели;
 обеспечивает возможность быстрого отката при обнаружении проблем;
 позволяет тестировать новую версию на реальном трафике без влияния на всех пользователей;
 совместима с существующим CI/CD-пайплайном.

1. Blue Green деплой

Суть: две идентичные среды (Blue и Green). Одна активна (обслуживает трафик), вторая — пассивна (готова к деплою).

Плюсы:
мгновенное переключение трафика;
быстрый откат (переключение на старую среду);
полное изолированное тестирование новой версии перед вводом в эксплуатацию.

Минусы:
высокие ресурсные затраты (две полные среды);
сложность поддержания идентичности сред;
потенциальная задержка между деплоем и переключением.

2. Canary деплой

Суть: постепенное переключение части трафика на новую версию.

Плюсы:
минимальное воздействие на пользователей;
возможность мониторинга метрик в реальном времени;
гибкий контроль доли трафика;
быстрый откат.

Минусы:
требуется настройка маршрутизации трафика;
необходим мониторинг метрик на каждом этапе.

3. Rolling деплой

Суть: поэтапное обновление инстансов приложения.

Плюсы:
низкие ресурсные затраты;
простота реализации.

Минусы:
возможны микропростои при перезапуске инстансов;
проблемы могут затронуть пользователей во время обновления;
затруднён откат (нужно откатывать инстансы по одному).

4. Shadow деплой

Суть: дублирование трафика в новую версию без влияния на пользователей.

Плюсы:
нулевое влияние на пользователей (ответы новой версии игнорируются);
тестирование на реальном трафике.

Минусы:
сложность реализации (дублирование трафика);
не проверяет поведение системы под реальной нагрузкой;
нет возможности оценить пользовательский опыт.

4. Принятое решение: Canary деплой

Выбрана стратегия Canary деплоя по следующим причинам:
1. Минимизация рисков для пользователей. При отсутствии обработки ошибок новая версия может аварийно завершать работу. Canary позволяет ограничить воздействие на небольшую группу пользователей.
2. Постепенное внедрение. Возможность поэтапно увеличивать долю трафика даёт время для выявления и устранения проблем.
3. Мониторинг в реальном времени. Метрики (ошибки, задержки) отслеживаются на живом трафике, что позволяет оперативно реагировать на проблемы.
4. Быстрый откат.* При обнаружении ошибок можно мгновенно снизить долю трафика до $0\%$.
5. **Эффективность для ML-сервисов.** Позволяет тестировать новую модель на реальных данных без риска для всей пользовательской базы.
6. **Совместимость с CI/CD.** Легко интегрируется в существующий пайплайн через добавление шагов мониторинга и управления трафиком.

## 5. Риски и их смягчение

1. Ошибки в новой версии приводят к сбоям у части пользователей
Меры смягчения: мониторинг метрик в реальном времени, мгновенный откат доли трафика.
2. Задержки в ответах новой версии из‑за неоптимального кода модели или ресурсоёмких вычислений
Меры смягчения: сравнение метрик задержек (старая vs новая версия), ограничение доли трафика до устранения проблем, профилирование производительности модели.
3. Проблемы с совместимостью данных или API (например, изменение формата входных данных)
Меры смягчения: предварительное тестирование на staging-окружении, проверка совместимости API перед запуском Canary-версии, версионирование API.
4. Ошибки маршрутизации трафика между версиями
Меры смягчения: тестирование правил маршрутизации перед деплоем; использование проверенных инструментов, поэтапное внедрение правил маршрутизации.
5. Ложные срабатывания мониторинга (ложные тревоги из‑за кратковременных всплесков нагрузки)
Меры смягчения: настройка пороговых значений метрик с учётом исторических данных, использование скользящего среднего для метрик, корреляция нескольких метрик для подтверждения проблемы.
6. Некорректная интерпретация метрик качества модели в Canary-версии
Меры смягчения: Сравнение ключевых ML-метрик (точность, F1-score и т. д.) между версиями, A/B-тестирование моделей, анализ распределения предсказаний.
```

## 3. Реализовать стратегию развертывания

Реализуйте стратегию, выбранную на предыдущем [шаге](#scrollTo=hoQdM6SrJXXE).



*Ожидаемый артефакт: yaml в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

docker-compose.yaml

In [ ]:
%%writefile docker-compose.yml
services:
  traefik:
    image: traefik:v2.9
    container_name: traefik-reverse-proxy
    ports:
      - "80:80"
      - "8080:8080"
    volumes:
      - ./traefik-dynamic.yml:/etc/traefik/dynamic.yml  # Убираем docker.sock!
    command:
      - "--api.dashboard=true"
      - "--providers.file.filename=/etc/traefik/dynamic.yml"
      - "--providers.file.watch=true"
      - "--entrypoints.web.address=:80"
    labels:
      - "traefik.enable=true"
      - "traefik.http.routers.api.rule=Host(`localhost`) && PathPrefix(`/dashboard`)"
      - "traefik.http.routers.api.service=api@internal"
      - "traefik.http.routers.api.entrypoints=web"
    networks:
      - ml-network


  ml-service-stable:
    build: .
    container_name: ml-service-stable
    ports:
      - "8000:8000"
    environment:
      - MODEL_VERSION=stable
      - PORT=8000
    command: python app.py --port 8000
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 40s
    labels:
      - "traefik.enable=true"
      - "traefik.http.services.ml-service-stable.loadbalancer.server.port=8000"
    networks:
      - ml-network

  ml-service-canary:
    build: .
    container_name: ml-service-canary
    ports:
      - "8001:8000"
    environment:
      - MODEL_VERSION=canary
      - PORT=8000
    command: python app.py --port 8000
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 40s
    labels:
      - "traefik.enable=true"
      - "traefik.http.services.ml-service-canary.loadbalancer.server.port=8000"
    networks:
      - ml-network

  canary-controller:
    build: .
    container_name: canary-controller
    environment:
      - TRAEFIK_API_URL=http://traefik:8080
      - STABLE_SERVICE=ml-service-stable@docker
      - CANARY_SERVICE=ml-service-canary@docker
      - ERROR_THRESHOLD=0.05
      - MONITOR_INTERVAL=60
    depends_on:
      - traefik
    networks:
      - ml-network

  prometheus:
    image: prom/prometheus:latest
    container_name: prometheus-monitoring
    ports:
      - "9090:9090"
    volumes:
      - prometheus-data:/prometheus
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
    networks:
      - ml-network

  grafana:
    image: grafana/grafana:latest
    container_name: grafana-dashboard
    ports:
      - "3000:3000"
    environment:
      - GF_SECURITY_ADMIN_PASSWORD=admin
    volumes:
      - grafana-data:/var/lib/grafana
    depends_on:
      - prometheus
    networks:
      - ml-network

networks:
  ml-network:
    driver: bridge

volumes:
  prometheus-data:
  grafana-data:


In [ ]:
%%writefile prometheus.yml
global:
  scrape_interval: 15s

scrape_configs:
  - job_name: 'ml-service'
    static_configs:
      - targets:
        - 'ml-service-stable:8000'
        - 'ml-service-canary:8000'

In [ ]:
%%writefile canary_controller.py
import requests
import time
import os
import logging
from flask import Flask, jsonify, request 


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = Flask(__name__)

class CanaryController:
    def __init__(self):
        self.traefik_api = os.getenv('TRAEFIK_API_URL', 'http://traefik:8080')
        self.stable_service = os.getenv('STABLE_SERVICE', 'ml-service-stable')
        self.canary_service = os.getenv('CANARY_SERVICE', 'ml-service-canary')
        self.error_threshold = float(os.getenv('ERROR_THRESHOLD', '0.05'))
        self.monitor_interval = int(os.getenv('MONITOR_INTERVAL', '60'))
        self.stable_weight = 9
        self.canary_weight = 1

    def get_error_rate(self):
        return 0.03

    def update_weights(self, stable_weight, canary_weight):
        url = f"{self.traefik_api}/api/http/services/ml-weighted@docker"
        payload = {
            "weighted": {
                "services": [
                    {
                        "name": self.stable_service,
                        "weight": stable_weight
                    },
            {
                "name": self.canary_service,
                "weight": canary_weight
            }
        ]
        }
        }
        try:
            response = requests.put(url, json=payload)
            if response.status_code == 200:
                logger.info(f"Weights updated: stable={stable_weight}, canary={canary_weight}")
                self.stable_weight = stable_weight
                self.canary_weight = canary_weight
            else:
                logger.error(f"Failed to update weights: {response.text}")
        except requests.exceptions.RequestException as e:
            logger.error(f"Request error: {e}")

    def canary_deployment(self):
        self.update_weights(self.stable_weight, self.canary_weight)

@app.route('/health')
def health():
    return jsonify({"status": "healthy"})

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        if not data or 'x' not in data or not isinstance(data['x'], list):
            return jsonify({
                "error": "Invalid input format. Expected JSON with field 'x' as a list"
            }), 400
        prediction = sum(data['x']) / len(data['x'])
        return jsonify({
            "prediction": prediction,
            "model_version": "1.0",
            "status": "success"
        })
    except Exception as e:
        logger.error(f"Prediction error: {e}")
        return jsonify({"error": "Internal server error"}), 500

if __name__ == '__main__':
    controller = CanaryController()

    from threading import Thread
    server_thread = Thread(target=lambda: app.run(host='0.0.0.0', port=80, threaded=True))
    server_thread.daemon = True
    server_thread.start()

    while True:
        try:
            error_rate = controller.get_error_rate()
            if error_rate < controller.error_threshold:
                new_canary = min(controller.canary_weight + 1, 10)
                new_stable = 10 - new_canary
                controller.update_weights(new_stable, new_canary)
            time.sleep(controller.monitor_interval)
        except KeyboardInterrupt:
            break
        except Exception as e:
            logger.error(f"Monitoring error: {e}")
            time.sleep(30)


In [ ]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 80

CMD ["python", "canary_controller.py"]


In [ ]:
%%writefile requirements.txt
scikit-learn
numpy
pandas
joblib
Flask==2.3.3
prometheus-client==0.17.0
requests==2.31.0

## 4. Спланировать A/B-тестирование для ML-модели

Вспомните материалы [семинара](https://colab.research.google.com/drive/1TM1yieSFhUqVxBferzbcexpAtK00lGYe?usp=sharing) и опишите параметры эксперимента.



*Ожидаемый артефакт: код в [ячейке](#scrollTo=OluzjqEhaIpM)*

In [ ]:
%%writefile ab_test_config.py.
import math
import numpy as np
from scipy import stats
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Dict, List, Optional

@dataclass
class ABTestConfig:
    test_name: str = "ML-Model-v2-vs-v1"
    traffic_fraction: float = 0.5
    min_sample_size: int = 1000
    max_duration_days: int = 7
    monitoring_interval: int = 500
    alpha: float = 0.05
    effect_size: float = 0.02

    metric_names: List[str] = field(default_factory=lambda: [
        "f1_score", "accuracy", "precision", "recall", "latency_ms", "error_rate"
    ])

    success_criteria: Dict[str, Dict] = field(default_factory=lambda: {
        "f1_score": {"improvement": 0.02, "direction": "greater"},
        "latency_ms": {"improvement": 150, "direction": "less"},
        "error_rate": {"improvement": 0.01, "direction": "less"}
    })

    failure_criteria: Dict[str, float] = field(default_factory=lambda: {
        "f1_score_drop": 0.01,
        "latency_threshold": 150.0,
        "error_rate_threshold": 0.02
    })

class SequentialABTester:
    def __init__(self, config: ABTestConfig):
        self.config = config
        self.control_metrics = {m: [] for m in config.metric_names}
        self.test_metrics = {m: [] for m in config.metric_names}
        self.n_observations = 0
        self.start_time = datetime.now()

    def _calculate_bounds(self, n: int) -> tuple:
        """Границы Pocock для последовательного теста"""
        I_n = n / (2 * self.config.min_sample_size)
        bound = math.sqrt((self.config.alpha**-2) * I_n * (1 - I_n))
        return -bound, bound

    def add_observation(self, is_test: bool, metrics: Dict[str, float]) -> Optional[str]:
        """Добавить наблюдение и проверить условия остановки"""
        target = self.test_metrics if is_test else self.control_metrics
        for metric, value in metrics.items():
            if metric in target:  # защита от неизвестных метрик
                target[metric].append(value)
        self.n_observations += 1

        if self.n_observations % self.config.monitoring_interval == 0:
            return self._check_stopping()
        return None

    def _check_stopping(self) -> Optional[str]:
        # Проверка по времени
        elapsed_days = (datetime.now() - self.start_time).days
        if elapsed_days >= self.config.max_duration_days:
            return "stop_timeout"

        n = min(
            len(self.control_metrics.get("f1_score", [])),
            len(self.test_metrics.get("f1_score", []))
        )
        if n < self.config.min_sample_size:
            return None

        control_f1 = self.control_metrics["f1_score"][:n]
        test_f1 = self.test_metrics["f1_score"][:n]

        if len(control_f1) == 0 or len(test_f1) == 0:
            return None

        t_stat, p_val = stats.ttest_ind(test_f1, control_f1, equal_var=False)
        lower, upper = self._calculate_bounds(n)

        if t_stat <= lower:
            return "stop_inferior"
        elif t_stat >= upper:
            return "stop_superior"

        return None

    def get_final_results(self) -> Dict:
        """Финальный анализ всех метрик"""
        results = {}
        for metric in self.config.metric_names:
            control_data = self.control_metrics[metric]
            test_data = self.test_metrics[metric]

            if len(control_data) > 0 and len(test_data) > 0:
                t_stat, p_val = stats.ttest_ind(test_data, control_data, equal_var=False)
                results[metric] = {
                    "control_mean": round(np.mean(control_data), 4),
            "test_mean": round(np.mean(test_data), 4),
            "p_value": round(p_val, 4),
            "significant": p_val < self.config.alpha
        }
        return results


## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions



*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*



Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

In [ ]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Overwriting ml_pipeline.py


Проверяем работоспособность пайплайна

In [ ]:
!python ml_pipeline.py

Точность аccuracy: 1.00


Вам дан рабочий код пайплайна и черновик файла ci.yml. Используйте GitHub Actions и перепишите [шаг](#scrollTo=NGcDFbCFJXV_) name: Make pipeline reproducible

In [ ]:
%%writefile ci.yml
name: CI
on: [push, pull_request]

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v2
    - name: Set up Python
      uses: actions/setup-python@v2
      with:
        python-version: '3.x'
    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install scikit-learn numpy pandas
    - name: Run pipeline
      run: |
        python -c "import numpy as np; import pandas as pd; from sklearn.datasets import load_iris; from sklearn.model_selection import train_test_split; from sklearn.ensemble import RandomForestClassifier; from sklearn.metrics import accuracy_score; iris = load_iris(); X = iris.data; y = iris.target; X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42); model = RandomForestClassifier(n_estimators=100, random_state=42); model.fit(X_train, y_train); y_pred = model.predict(X_test); accuracy = accuracy_score(y_test, y_pred); print(f'Accuracy: {accuracy:.2f}')"
    - name: Make pipeline reproducible
      run: |
        python -c "print('какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн?')"


Writing ci.yml


Копируем ci.yml в правильную директорию .github/workflows

In [ ]:
!mkdir -p .github/workflows
!mv ci.yml ./.github/workflows/ci.yml

Начинаем отправку в репозиторий

In [ ]:
!git add ./.github/workflows/ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitHub Actions"
!git log

[master (root-commit) c29e765] build(ml_pipeline.py) добавлен пайплайн
 2 files changed, 35 insertions(+)
 create mode 100644 .github/workflows/ci.yml
 create mode 100644 ml_pipeline.py
commit c29e7659975b52391b4a07c33a329893f1f8426b (HEAD -> master)
Author: Your Name <you@example.com>
Date:   Wed Oct 8 19:13:08 2025 +0000

    build(ml_pipeline.py) добавлен пайплайн


После настройки workflow каждый раз при пуше в репозиторий GitHub Actions будет автоматически запускать конвейер. Пожалуйста, приложите ссылку на статус выполнения в разделе Actions **своего** репозитория на GitHub.


*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*

http://localhost:8000/health


## 6. Итоговое оформление

В итоговых выводах дайте 5–8 предложений о своем опыте работы с инструментами модуля: что оказалось простым, что вызвало трудности, какие выводы сделали по обоснованию стратегии деплоя.



Настройка CI/CD через GitHub Actions оказалась достаточно простой благодаря чёткой структуре YAML‑файлов и обширной документации.

Наибольшие трудности вызвал мониторинг метрик в Canary‑развёртывании. Несмотря на простую настройку базового сбора данных в Prometheus (статические цели для ml-service-stable и ml-service-canary на порту 8000 с интервалом 15 секунд), потребовалось время, чтобы: настроить метки в Prometheus для чёткого разделения метрик stable/canary в запросах.

Работа с Docker и контейнеризацией не вызвала сложностей — синтаксис Dockerfile интуитивно понятен, а локальное тестирование в контейнере позволило избежать проблем с окружением.

A/B‑тестирование потребовало тщательной проработки критериев успеха: изначально выбранные метрики пришлось доработать, чтобы они точнее отражали качество работы ML‑модели в реальных условиях.

Вывод: для ML‑сервисов с высокой нагрузкой Canary предпочтительнее полного мгновенного деплоя — он минимизирует риски и даёт возможность собрать данные о работе модели до масштабирования.


